In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    r"D:\Course\python\energy forecasting\energy_env\data\raw\time_series_60min_singleindex.csv",  # ✅ raw string
    parse_dates=["utc_timestamp"]
)

print(df.shape)
df.head()



(50401, 300)


,utc_timestamp,cet_cest_timestamp,AT_load_actual_entsoe_transparency,AT_load_forecast_entsoe_transparency,AT_price_day_ahead,AT_solar_generation_actual,AT_wind_onshore_generation_actual,BE_load_actual_entsoe_transparency,BE_load_forecast_entsoe_transparency,BE_solar_generation_actual,...,SI_load_actual_entsoe_transparency,SI_load_forecast_entsoe_transparency,SI_solar_generation_actual,SI_wind_onshore_generation_actual,SK_load_actual_entsoe_transparency,SK_load_forecast_entsoe_transparency,SK_solar_generation_actual,SK_wind_onshore_generation_actual,UA_load_actual_entsoe_transparency,UA_load_forecast_entsoe_transparency
0,2014-12-31 23:00:00+00:00,2015-01-01T00:00:00+0100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-01-01 00:00:00+00:00,2015-01-01T01:00:00+0100,5946.0,6701.0,35.0,NaN,69.0,9484.0,9897.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-01-01 01:00:00+00:00,2015-01-01T02:00:00+0100,5726.0,6593.0,45.0,NaN,64.0,9152.0,9521.0,NaN,...,1045.47,816.0,NaN,1.17,2728.0,2860.0,3.8,NaN,NaN,NaN
3,2015-01-01 02:00:00+00:00,2015-01-01T03:00:00+0100,5347.0,6482.0,41.0,NaN,65.0,8799.0,9135.0,NaN,...,1004.79,805.0,NaN,1.04,2626.0,2810.0,3.8,NaN,NaN,NaN
4,2015-01-01 03:00:00+00:00,2015-01-01T04:00:00+0100,5249.0,6454.0,38.0,NaN,64.0,8567.0,8909.0,NaN,...,983.79,803.0,NaN,1.61,2618.0,2780.0,3.8,NaN,NaN,NaN


In [2]:
KEEP_COLS = [
    "utc_timestamp",
    "DE_load_actual_entsoe_transparency",
    "DE_load_forecast_entsoe_transparency",
    "DE_solar_generation_actual",
    "DE_wind_generation_actual",
    "DE_wind_onshore_generation_actual",
    "DE_wind_offshore_generation_actual",
]

df = df[KEEP_COLS]


In [3]:
df = df.rename(columns={
    "utc_timestamp": "time",
    "DE_load_actual_entsoe_transparency": "load",
    "DE_load_forecast_entsoe_transparency": "load_forecast",
    "DE_solar_generation_actual": "solar",
    "DE_wind_generation_actual": "wind",
    "DE_wind_onshore_generation_actual": "wind_onshore",
    "DE_wind_offshore_generation_actual": "wind_offshore"
})


In [4]:
# Sort and set datetime index
df = df.sort_values("time")
df = df.set_index("time")

# Time-based interpolation
df = df.interpolate(method="time")

# Drop remaining missing values
df = df.dropna()

# Reset index back to column
df = df.reset_index()

print("Missing values after cleaning:")
print(df.isna().sum())


Missing values after cleaning:
time             0
load             0
load_forecast    0
solar            0
wind             0
wind_onshore     0
wind_offshore    0
dtype: int64


In [5]:
df["hour"] = df["time"].dt.hour
df["day_of_week"] = df["time"].dt.weekday
df["day_of_month"] = df["time"].dt.day
df["month"] = df["time"].dt.month
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)


In [6]:
import sys
print(sys.executable)


d:\Course\python\energy forecasting\energy_env\Scripts\python.exe


In [ ]:
# !"{sys.executable}" -m pip install holidays


  Using cached holidays-0.88-py3-none-any.whl.metadata (50 kB)
Using cached holidays-0.88-py3-none-any.whl (1.3 MB)


In [7]:
import holidays
print(holidays.__version__)


0.88


In [8]:
import holidays

de_holidays = holidays.Germany()

df["is_holiday"] = df["time"].dt.date.apply(
    lambda x: 1 if x in de_holidays else 0
)


In [9]:
TARGET = "load"

PAST_OBSERVED = [
    "load",
    "solar",
    "wind",
    "wind_onshore",
    "wind_offshore"
]

KNOWN_FUTURE = [
    "hour",
    "day_of_week",
    "day_of_month",
    "month",
    "is_weekend",
    "is_holiday",
    "load_forecast"
]


In [10]:
REQUIRED_COLS = ["time", TARGET] + PAST_OBSERVED[1:] + KNOWN_FUTURE

missing = [c for c in REQUIRED_COLS if c not in df.columns]
print("Missing columns:", missing)


Missing columns: []


In [11]:
FINAL_COLS = (
    ["time"] +
    [TARGET] +
    PAST_OBSERVED[1:] +   # remove duplicate load
    KNOWN_FUTURE
)

df_final = df[FINAL_COLS]


In [13]:
df_final.to_csv(
    "D:/Course/python/energy forecasting/energy_env/data/processed/germany_hourly_processed.csv",
    index=False
)
df.head()

,time,load,load_forecast,solar,wind,wind_onshore,wind_offshore,hour,day_of_week,day_of_month,month,is_weekend,is_holiday
0,2015-01-01 07:00:00+00:00,41133.0,42522.0,71.0,10208.0,9683.0,525.0,7,3,1,1,0,1
1,2015-01-01 08:00:00+00:00,42963.0,45020.0,773.0,10029.0,9502.0,527.0,8,3,1,1,0,1
2,2015-01-01 09:00:00+00:00,45088.0,47101.0,2117.0,10550.0,10025.0,525.0,9,3,1,1,0,1
3,2015-01-01 10:00:00+00:00,47013.0,49603.0,3364.0,11390.0,10862.0,528.0,10,3,1,1,0,1
4,2015-01-01 11:00:00+00:00,48159.0,49910.0,4198.0,12103.0,11575.0,528.0,11,3,1,1,0,1


In [17]:
df_check = pd.read_csv(
   "D:/Course/python/energy forecasting/energy_env/data/processed/germany_hourly_processed.csv",
    parse_dates=["time"]
)

print(df_check.shape)
df_check.head()


(50393, 13)


,time,load,solar,wind,wind_onshore,wind_offshore,hour,day_of_week,day_of_month,month,is_weekend,is_holiday,load_forecast
0,2015-01-01 07:00:00+00:00,41133.0,71.0,10208.0,9683.0,525.0,7,3,1,1,0,1,42522.0
1,2015-01-01 08:00:00+00:00,42963.0,773.0,10029.0,9502.0,527.0,8,3,1,1,0,1,45020.0
2,2015-01-01 09:00:00+00:00,45088.0,2117.0,10550.0,10025.0,525.0,9,3,1,1,0,1,47101.0
3,2015-01-01 10:00:00+00:00,47013.0,3364.0,11390.0,10862.0,528.0,10,3,1,1,0,1,49603.0
4,2015-01-01 11:00:00+00:00,48159.0,4198.0,12103.0,11575.0,528.0,11,3,1,1,0,1,49910.0
